# Week 4 – Predictive Modeling and Optimization for Logistics Operations

**Target:** Predict shipment delivery time and use predictions to support operational decisions.

**Dataset:** Simulated logistics shipment data.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/processed/cleaned_logistics_data.csv")
print("Shape:", df.shape)

## 1. Define Features and Target

In [ ]:
X = df.drop(columns=["Delivery_Time_Hours"])
y = df["Delivery_Time_Hours"]

numeric_features = [
    "Distance_km",
    "Package_Weight_kg",
    "Stops_Count",
    "Warehouse_Delay_Min",
    "Driver_Experience_Years"
]

categorical_features = [
    "Traffic_Level",
    "Weather",
    "Vehicle_Type",
    "Delivery_Priority"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 2. Train Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=8, random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

trained_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe

print("Models trained:", list(trained_models.keys()))

## 3. Evaluate Models

In [ ]:
results = []

for name, model in trained_models.items():
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2"]
).sort_values("RMSE")

display(results_df)

results_df.to_csv("../results/model_comparison.csv", index=False)

## 4. Five-Fold Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in trained_models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    cv_rmse = -scores.mean()
    cv_results.append([name, cv_rmse])

cv_df = pd.DataFrame(
    cv_results,
    columns=["Model", "CV_RMSE"]
).sort_values("CV_RMSE")

display(cv_df)
cv_df.to_csv("../results/cross_validation_results.csv", index=False)

## 5. Random Forest Hyperparameter Tuning

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [8, 12, 16, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

print("Best parameters:")
print(search.best_params_)

## 6. Prediction and Delivery-Risk Classification

In [ ]:
predicted = best_model.predict(X_test)

decision_df = X_test.copy()
decision_df["Predicted_Delivery_Hours"] = predicted

SLA_HOURS = 10

decision_df["Risk_Level"] = np.select(
    [
        decision_df["Predicted_Delivery_Hours"] > SLA_HOURS + 3,
        decision_df["Predicted_Delivery_Hours"] > SLA_HOURS
    ],
    ["High", "Medium"],
    default="Low"
)

priority_shipments = decision_df[
    decision_df["Risk_Level"] == "High"
]

print("Risk distribution:")
print(decision_df["Risk_Level"].value_counts())

display(priority_shipments.head())

decision_df.to_csv("../results/predicted_delivery_risk.csv", index=False)

## 7. Operational Optimization Logic

In [ ]:
optimization_actions = {
    "High": [
        "Prioritize route review",
        "Consider experienced driver allocation",
        "Check vehicle capacity",
        "Review delivery slot",
        "Consider contingency capacity"
    ],
    "Medium": [
        "Monitor shipment",
        "Review route and traffic conditions"
    ],
    "Low": [
        "Proceed under normal operating plan"
    ]
}

for risk, actions in optimization_actions.items():
    print(f"\n{risk} Risk:")
    for action in actions:
        print("-", action)

## 8. Optimization Objective

The broader optimization objective is to minimize total operational cost while satisfying
delivery and resource constraints.

Conceptually:

**Total Cost = Vehicle Cost + Driver Overtime Cost + Fuel Cost + Late Delivery Penalty**

Subject to vehicle capacity, driver working-hour, shipment assignment, priority,
and service-level constraints.

## 9. Important Limitation

The dataset in this repository is simulated. Model performance obtained from simulated data
should not be presented as real company performance or guaranteed real-world accuracy.